In [1]:
import h5py
import numpy as np
from opt_einsum import contract

def jackknife(x):
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)  

F_or_B = "forward"
gamma = "45"
q_tag = "q0_0_0"
hyp_tag = "0-1-2-3-4-5-6-7-8"
local = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis"
FF_path = f"{local}/FF_symmetric_{q_tag}_hyp{hyp_tag}_fb_tins18_ncfg799.h5"
pt2_path = f"{local}/2pt_N40_rho3.25_G{gamma}_ez_momfrac0p6_symmetric_{q_tag}_fb_tsep18_ncfg799.h5"
Ls = 32
SRC_PHASE_SIGN = -1    # in my convention pi = pf + q, phase for FF is iq(x-xsrc) so -1 
WL_PHASE_SIGN = +1     # Fix the centering of Wilson line
FF_file = h5py.File(FF_path, "r")
FF_cfg = FF_file[f"FF_{F_or_B}"]                         # [cfg, tsrc, munu, rhosig, tgf, w, q, tau] = (799, 8, 6, 6, 5, 10, 1, 18)
q_list = FF_file["q_list"][:]                            # here we use only [[0,0,0]]
w_list = FF_file["w_list"][:]                            
print(FF_cfg.shape)
pt2_file = h5py.File(pt2_path, "r")
pt2_cfg = pt2_file[f"pt2_f_{F_or_B}"][:]                   # [cfg, tsrc, xsrc, ysrc, zsrc, pf, tsep] = (799, 8, 4, 4, 8, 7, 18)
cfg_list = pt2_file["cfg_list"][:]                       
print(pt2_cfg.shape)
pt2_f_avg = contract(pt2_cfg, ["cfg", "tsrc", "xsrc", "ysrc", "zsrc", "pf", "tsep"], ["cfg", "pf", "tsep"]) / (8 * 4 * 4 * 8)   # no source phase
pt2_f_jk = jackknife(pt2_f_avg)                   # [jk, pf, tsep] = (799, 7, 18)
#In my FF production code, Wilson line is not from -w/2 to w/2 so we need an extra phase factor to restore the convention (exactly)
wl_phase = np.exp(WL_PHASE_SIGN * 1j * np.pi / Ls * contract(w_list,["w"], q_list[:, 2],  ["q"], ["w", "q"]))                                  # [w, q] = (10, 1)
TXTX = contract(FF_cfg[:, :, 3, 3,:,:,:,:], ["cfg", "tsrc", "tgf", "w", "q", "tau"],wl_phase,["w", "q"], ["cfg", "tsrc", "tgf", "w", "q", "tau"])     # [cfg, tsrc, tgf, w, q, tau] = (799, 8, 5, 10, 1, 18)
TYTY = contract(FF_cfg[:, :, 4, 4,:,:,:,:], ["cfg", "tsrc", "tgf", "w", "q", "tau"],wl_phase,["w", "q"],["cfg", "tsrc", "tgf", "w", "q", "tau"])
XYXY = contract(FF_cfg[:, :, 0, 0,:,:,:,:], ["cfg", "tsrc", "tgf", "w", "q", "tau"],wl_phase,["w", "q"],["cfg", "tsrc", "tgf", "w", "q", "tau"])
TXTXpTYTY = TXTX + TYTY
TXTXpTYTYm2XYXY = TXTXpTYTY - 2 * XYXY
TXTXpTYTYp2XYXY = TXTXpTYTY + 2 * XYXY
operators = [TXTX, TYTY, XYXY, TXTXpTYTY, TXTXpTYTYm2XYXY,TXTXpTYTYp2XYXY]

shift = ((cfg_list - 204) // 6 * 3) % Ls                               # [cfg]   spatial source shift of each configuration
x_pos = (np.arange(0, 32, 8)[None, :] + shift[:, None]) % Ls           # [cfg, xsrc]
y_pos = (np.arange(0, 32, 8)[None, :] + shift[:, None]) % Ls           # [cfg, ysrc]
z_pos = (np.arange(0, 32, 4)[None, :] + shift[:, None]) % Ls           # [cfg, zsrc]
# [cfg, xsrc, ysrc, zsrc, q]
print(q_list)
print(q_list[:,0])
print((x_pos[:, :, None, None, None] * q_list[:, 0]).shape)     # (799, 4, 1, 1, 1)
print((x_pos[:, :, None, None, None] * q_list[:, 0])[0,0])
q_dot_x = (x_pos[:, :, None, None, None] * q_list[:, 0] + y_pos[:, None, :, None, None] * q_list[:, 1] + z_pos[:, None, None, :, None] * q_list[:, 2])  
src_phase = np.exp(SRC_PHASE_SIGN * 1j * 2 * np.pi / Ls * q_dot_x)     # [cfg, xsrc, ysrc, zsrc, q]

#FF phase can be contracted with pt2 first
pt2_src = contract(src_phase, ["cfg", "xsrc", "ysrc", "zsrc", "q"],
                   pt2_cfg,   ["cfg", "tsrc", "xsrc", "ysrc", "zsrc", "pf", "tsep"],
                              ["cfg", "tsrc", "pf", "tsep", "q"]) / (4 * 4 * 8)   # [cfg, tsrc, pf, tsep, q]


#Calculate the connected piece
pt2FF = []
for iop, op in enumerate(operators):
    print(iop)
    connected_term = contract(pt2_src, ["cfg", "tsrc", "pf", "tsep", "q"],
                              op,      ["cfg", "tsrc", "tgf", "w", "q", "tau"],
                                       ["cfg", "tgf", "w", "q", "tsep", "tau", "pf"])
    pt2FF.append(connected_term / 8)                     # [cfg, tgf, w, q, tsep, tau, pf]          


#Calculate the disconnected piece using my old method
#First <2pt>
pt2_avg = contract(pt2_src, ["cfg", "tsrc", "pf", "tsep", "q"],["cfg", "pf", "tsep", "q"]) / 8  
#Then FF operator
op_avg = []
for iop, op in enumerate(operators):
    op_avg.append(contract(op, ["cfg", "tsrc", "tgf", "w", "q", "tau"],["cfg", "tgf", "w", "q", "tau"]) / 8)

pt2_jk = jackknife(pt2_avg)                       
op_jk = [jackknife(a) for a in op_avg]            
pt2FF_jk = [jackknife(a) for a in pt2FF] 
del pt2FF
pt3_vac_subtracted = []
for iop in range(len(operators)):
    vac_jk = contract(pt2_jk,     ["jk", "pf", "tsep", "q"],
                      op_jk[iop], ["jk", "tgf", "w", "q", "tau"],
                                  ["jk", "tgf", "w", "q", "tsep", "tau", "pf"])   # [jk, tgf, w, q, tsep, tau, pf] = (799, 5, 10, 1, 18, 18, 7)
    pt3_vac_subtracted.append(pt2FF_jk[iop] - vac_jk)                              # [jk, tgf, w, q, tsep, tau, pf]

op_names = ["TXTX", "TYTY", "XYXY", "TXTXpTYTY", "TXTXpTYTYm2XYXY","TXTXpTYTYp2XYXY"]
pf_list = pt2_file["pf_list"][:]                       # [pf, 3] = (7, 3)
hyp_list = FF_file["hyp_list"][:]                      # [hyp] = (9,)
out_path = (f"{local}/"
            f"pt3_jk_method1_G{gamma}_{F_or_B}_{q_tag}_hyp{hyp_tag}_src{SRC_PHASE_SIGN:+d}_wl{WL_PHASE_SIGN:+d}.h5")
with h5py.File(out_path, "w") as f:
    for iop, name in enumerate(op_names):
        f.create_dataset(f"pt3_jk/{name}", data=pt3_vac_subtracted[iop])   # [jk, tgf, w, q, tsep, tau, pf] = (799, 5, 10, 1, 18, 18, 7)
    f.create_dataset("pt2_jk", data=pt2_jk)                                # [jk, pf, tsep, q] = (799, 7, 18, 1)
    f.create_dataset("pt2_f_jk", data=pt2_f_jk)                            # [jk, pf, tsep]
    f.create_dataset("operator_names", data=np.array(op_names, dtype="S"))
    f.attrs["forward_or_backward"] = F_or_B
    f.create_dataset("pf_list", data=pf_list)
    f.create_dataset("hyp_list", data=hyp_list)
    f.create_dataset("w_list", data=w_list)
    f.create_dataset("q_list", data=q_list)
    f.create_dataset("cfg_list", data=cfg_list)
    f.attrs["dim_pt3_jk"] = "jk, hyp, w, q, tsep, tau, pf"
    f.attrs["smearing"] = FF_file.attrs["smearing"]
    f.attrs["dim_pt2_jk"] = "jk, pf, tsep, q"
    f.attrs["vacuum_subtraction"] = "method 1: average over sources, jackknife over cfgs, multiply <C2><O>, subtract per replicate"
    f.attrs["src_phase"] = f"exp({SRC_PHASE_SIGN:+d} i q.x_src), x_src = shifted source positions"
    f.attrs["wl_phase"] = f"exp({WL_PHASE_SIGN:+d} i pi q_z w / Ls), applied to the operator"
    f.attrs["jackknife"] = "delete-one, replicate k = mean over all cfgs except cfg_list[k]"
print("saved", out_path)
del pt3_vac_subtracted

#New method given by claude. checked to give consistent result with my old method
#multiply jackknifed 2pt and FF per-tsrc then average tsrc
pt2_src_jk = jackknife(pt2_src)                    # [jk, tsrc, pf, tsep, q] = (799, 8, 7, 18, 1)
op_src_jk = [jackknife(op) for op in operators]    # each [jk, tsrc, tgf, w, q, tau] = (799, 8, 5, 10, 1, 18)
pt3_vac_subtracted_m2 = []
for iop in range(len(operators)):
    vac_jk = contract(pt2_src_jk,    ["jk", "tsrc", "pf", "tsep", "q"],
                      op_src_jk[iop], ["jk", "tsrc", "tgf", "w", "q", "tau"],
                                      ["jk", "tgf", "w", "q", "tsep", "tau", "pf"]) / 8   # [jk, tgf, w, q, tsep, tau, pf]
    pt3_vac_subtracted_m2.append(pt2FF_jk[iop] - vac_jk)
out_path = (f"{local}/"
            f"pt3_jk_method2_G{gamma}_{F_or_B}_{q_tag}_hyp{hyp_tag}_src{SRC_PHASE_SIGN:+d}_wl{WL_PHASE_SIGN:+d}.h5")

with h5py.File(out_path, "w") as f:
    for iop, name in enumerate(op_names):
        f.create_dataset(f"pt3_jk/{name}", data=pt3_vac_subtracted_m2[iop])   # [jk, tgf, w, q, tsep, tau, pf]
    f.create_dataset("pt2_jk", data=pt2_jk)
    f.create_dataset("pt2_f_jk", data=pt2_f_jk)                            # [jk, pf, tsep]
    f.attrs["forward_or_backward"] = F_or_B
    f.create_dataset("operator_names", data=np.array(op_names, dtype="S"))
    f.create_dataset("pf_list", data=pf_list)
    f.create_dataset("hyp_list", data=hyp_list)
    f.create_dataset("w_list", data=w_list)
    f.create_dataset("q_list", data=q_list)
    f.create_dataset("cfg_list", data=cfg_list)
    f.attrs["dim_pt3_jk"] = "jk, hyp, w, q, tsep, tau, pf"
    f.attrs["smearing"] = FF_file.attrs["smearing"]
    f.attrs["dim_pt2_jk"] = "jk, pf, tsep, q"
    f.attrs["vacuum_subtraction"] = "method 2: jackknife over cfgs per source, multiply per source, average over sources"
    f.attrs["src_phase"] = f"exp({SRC_PHASE_SIGN:+d} i q.x_src), x_src = shifted source positions"
    f.attrs["wl_phase"] = f"exp({WL_PHASE_SIGN:+d} i pi q_z w / Ls), applied to the operator"
    f.attrs["jackknife"] = "delete-one, replicate k = mean over all cfgs except cfg_list[k]"
print("saved", out_path)
del pt3_vac_subtracted_m2
#My alternative method. We decided to discard this method a few months ago. Now claude has shown it's wrong.
#No formula needed to disprove it. When we have only one tsrc, this gives exactly zero. So it's wrong.
pt3_vac_subtracted_m3 = []
for iop in range(len(operators)):
    vac_cfg = contract(pt2_avg,     ["cfg", "pf", "tsep", "q"],
                       op_avg[iop], ["cfg", "tgf", "w", "q", "tau"],
                                    ["cfg", "tgf", "w", "q", "tsep", "tau", "pf"])   # [cfg, tgf, w, q, tsep, tau, pf]   product on each cfg
    pt3_vac_subtracted_m3.append(pt2FF_jk[iop] - jackknife(vac_cfg))                 # [jk, tgf, w, q, tsep, tau, pf]
out_path = (f"{local}/"
            f"pt3_jk_method3_G{gamma}_{F_or_B}_{q_tag}_hyp{hyp_tag}_src{SRC_PHASE_SIGN:+d}_wl{WL_PHASE_SIGN:+d}.h5")

with h5py.File(out_path, "w") as f:
    for iop, name in enumerate(op_names):
        f.create_dataset(f"pt3_jk/{name}", data=pt3_vac_subtracted_m3[iop])   # [jk, tgf, w, q, tsep, tau, pf]
    f.create_dataset("pt2_jk", data=pt2_jk)
    f.create_dataset("pt2_f_jk", data=pt2_f_jk)                            # [jk, pf, tsep]
    f.create_dataset("operator_names", data=np.array(op_names, dtype="S"))
    f.attrs["forward_or_backward"] = F_or_B
    f.create_dataset("pf_list", data=pf_list)
    f.create_dataset("hyp_list", data=hyp_list)
    f.create_dataset("w_list", data=w_list)
    f.create_dataset("q_list", data=q_list)
    f.create_dataset("cfg_list", data=cfg_list)
    f.attrs["dim_pt3_jk"] = "jk, hyp, w, q, tsep, tau, pf"
    f.attrs["smearing"] = FF_file.attrs["smearing"]
    f.attrs["dim_pt2_jk"] = "jk, pf, tsep, q"
    f.attrs["vacuum_subtraction"] = "method 3: multiply per-cfg source averages on each cfg, then jackknife (biased by ~1/n_tsrc of the signal; comparison only)"
    f.attrs["src_phase"] = f"exp({SRC_PHASE_SIGN:+d} i q.x_src), x_src = shifted source positions"
    f.attrs["wl_phase"] = f"exp({WL_PHASE_SIGN:+d} i pi q_z w / Ls), applied to the operator"
    f.attrs["jackknife"] = "delete-one, replicate k = mean over all cfgs except cfg_list[k]"
print("saved", out_path)

(799, 8, 6, 6, 9, 10, 1, 18)
(799, 8, 4, 4, 8, 7, 18)
[[0 0 0]]
[0]
(799, 4, 1, 1, 1)
[[[0]]]
0
1
2
3
4
5
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/pt3_jk_method1_G45_backward_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/pt3_jk_method2_G45_backward_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/pt3_jk_method3_G45_backward_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5


In [7]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    print(n)
    data = np.moveaxis(data, jk_axis, 0)
    print(f"data has shape {data.shape}")
    mean = np.mean(data, axis=0)
    print(f"mean has shape {mean.shape}")
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err

GAMMA = "G45"           # "G45" or "G5": must match the tag in the pt3_jk file names
method = 1
local = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis"
q_tag = "q0_0_0"
hyp_tag = "0-1-2-3-4-5-6-7-8"
paths = {d: f"{local}/pt3_jk_method{method}_{GAMMA}_{d}_{q_tag}_hyp{hyp_tag}_src-1_wl+1.h5" for d in ("forward", "backward")}
operator = "TXTXpTYTYm2XYXY"
hyp_plot = 1
w_plot_list = [0, 2, 4, 6, 8]
tsep_plot_list = [5, 6, 7, 8, 9, 10]
tsep_tag = f"tsep{tsep_plot_list[0]}-{tsep_plot_list[-1]}"
plot_dir = f"{local}/ratio_plots"
ratio_dir = f"{local}/ratio_data"
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(ratio_dir, exist_ok=True)

# ---------- read forward and backward 3pt / 2pt jackknife lists ----------
pt3, pt2 = {}, {}
for d in ("forward", "backward"):
    with h5py.File(paths[d], "r") as f:
        pt3[d] = f[f"pt3_jk/{operator}"][:]      # [jk, hyp, w, q, tsep, tau, pf] = (799, 9, 10, 1, 18, 18, 7)
        pt2[d] = f["pt2_f_jk"][:]                # [jk, pf, tsep] = (799, 7, 18)   plain C2(pf)
        hyp_list = list(f["hyp_list"][:])
        w_list = list(f["w_list"][:])
        pf_list = f["pf_list"][:]
        q_list = f["q_list"][:]
        cfg_list = f["cfg_list"][:]
        vacuum_method = str(f.attrs["vacuum_subtraction"])
        
pt3["fb"] = 0.5 * (pt3["forward"] + pt3["backward"])     # replicate by replicate: same cfg_list, same left-out cfg
pt2["fb"] = 0.5 * (pt2["forward"] + pt2["backward"])     # = folded 2pt

# ---------- ratio on every jackknife replicate, saved per direction ----------
ratio_mean, ratio_err = {}, {}
for d in ("forward", "backward", "fb"):
    c2 = pt2[d].transpose(0, 2, 1)                                      # [jk, tsep, pf]
    ratio_jk = pt3[d].real / c2[:, None, None, None, :, None, :].real   # [jk, hyp, w, q, tsep, tau, pf]
    ratio_mean[d], ratio_err[d] = jk(ratio_jk)                          # each [hyp, w, q, tsep, tau, pf]
    print(ratio_mean[d].shape)
    out = f"{ratio_dir}/ratio_{GAMMA}_method{method}_{d}_{operator}_{q_tag}_hyp{hyp_tag}_src-1_wl+1.h5"
    with h5py.File(out, "w") as f:
        f.create_dataset("ratio_jk", data=ratio_jk)                     # [jk, hyp, w, q, tsep, tau, pf]   tsep, tau = time index; tau > tsep is not a ratio
        f.create_dataset("ratio_mean", data=ratio_mean[d])              # [hyp, w, q, tsep, tau, pf]
        f.create_dataset("ratio_err", data=ratio_err[d])                # [hyp, w, q, tsep, tau, pf]
        f.create_dataset("hyp_list", data=np.array(hyp_list, dtype=np.int64))
        f.create_dataset("w_list", data=np.array(w_list, dtype=np.int64))
        f.create_dataset("pf_list", data=pf_list)
        f.create_dataset("q_list", data=q_list)
        f.create_dataset("cfg_list", data=cfg_list)
        f.attrs["dim_ratio_jk"] = "jk, hyp, w, q, tsep, tau, pf"
        f.attrs["ratio_definition"] = "R = Re C3(tsep,tau) / Re C2(pf,tsep)"
        f.attrs["operator"] = operator
        f.attrs["gamma"] = GAMMA
        f.attrs["vacuum_method"] = method
        f.attrs["vacuum_subtraction"] = vacuum_method
        f.attrs["direction"] = d + ("" if d != "fb" else ": 0.5 * (forward + backward) of the 3pt and 2pt jackknife lists, then the ratio")
        f.attrs["pt3_source"] = paths.get(d, f"{paths['forward']} + {paths['backward']}")
    print(f"saved {out}")

# ---------- plots: one figure per pz, one panel per w: forward vs forward+backward ----------
style = {"forward": dict(marker="o", ls="none", ms=4),
         "fb":      dict(marker="o", mfc="none", ls="none", ms=5)}
offset = {"forward": -0.08, "fb": +0.08}
ihyp = hyp_list.index(hyp_plot)
for ipf, pf in enumerate(pf_list):
    plt.figure(figsize=(4.2 * len(w_plot_list), 4.5))
    for icol, w in enumerate(w_plot_list):
        iw = w_list.index(w)
        plt.subplot(1, len(w_plot_list), icol + 1)
        for itsep, tsep in enumerate(tsep_plot_list):
            tau = np.arange(0, tsep + 1)
            x = tau - tsep / 2
            for d in ("forward", "fb"):
                plt.errorbar(x + offset[d], ratio_mean[d][ihyp, iw, 0, tsep, tau, ipf],
                             yerr=ratio_err[d][ihyp, iw, 0, tsep, tau, ipf],
                             capsize=2, color=f"C{itsep}",
                             label=f"tsep={tsep}" if d == "forward" else None, **style[d])
        plt.title(f"pz={pf[2]}  w={w}   filled: forward,  hollow: forward+backward", fontsize=9)
        plt.xlabel("tau - tsep/2")
        plt.autoscale(enable=False, axis="y")
        plt.ylim(-0.5, 0.1)
        if icol == 0:
            plt.ylabel("R = C3 / C2")
        if icol == len(w_plot_list) - 1:
            plt.legend(fontsize=7)
    plt.suptitle(f"{operator}   {GAMMA}   method {method}   hyp{hyp_plot}   {q_tag}", fontsize=12)
    plt.tight_layout()
    out = f"{plot_dir}/ratio_{GAMMA}_method{method}_fwd_vs_fb_{operator}_hyp{hyp_plot}_pz{pf[2]}_{tsep_tag}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"saved {out}")

799
data has shape (799, 9, 10, 1, 18, 18, 7)
mean has shape (9, 10, 1, 18, 18, 7)
(9, 10, 1, 18, 18, 7)
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/ratio_data/ratio_G45_method1_forward_TXTXpTYTYm2XYXY_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5
799
data has shape (799, 9, 10, 1, 18, 18, 7)
mean has shape (9, 10, 1, 18, 18, 7)
(9, 10, 1, 18, 18, 7)
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/ratio_data/ratio_G45_method1_backward_TXTXpTYTYm2XYXY_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5
799
data has shape (799, 9, 10, 1, 18, 18, 7)
mean has shape (9, 10, 1, 18, 18, 7)
(9, 10, 1, 18, 18, 7)
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/ratio_data/ratio_G45_method1_fb_TXTXpTYTYm2XYXY_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/ratio_plots/ratio_G45_method1_fwd_vs_fb_TXTXpTYTYm2XYXY_hyp1_pz0_tsep5-10.png
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analy

In [60]:
%%writefile fit_ratio_hyp.py
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import gvar as gv
import h5py
import lsqfit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

"""
In this code we don't really loop over w to fit independently. We multiply off-w elements in the covariance matrix by a factor.
When this factor is zero, it is effectively the same as fitting different w separately.
Input: one ratio file per (gamma, vacuum method, direction) holding R = C3/C2 for all hyp, pf, q, [jk, hyp, w, q, tsep, tau, pf].
"""

SCRIPT_DIR = Path(__file__).resolve().parent
frame = "symmetric"
operator = "TXTXpTYTYm2XYXY"

GAMMA = "G45"                  # "G45" or "G5"
VACUUM_METHOD = 1              # 1, 2 or 3
DIRECTION = "fb"          # "forward", "backward" or "fb" (forward+backward averaged)
FOLD_2PT = False               # read the gap prior from the folded 2pt fit

LOCAL = Path("/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis")
q_tag = "q0_0_0"
hyp_tag = "0-1-2-3-4-5-6-7-8"
RATIO_FILE = LOCAL / "ratio_data" / (f"ratio_{GAMMA}_method{VACUUM_METHOD}_{DIRECTION}_{operator}"
                                     f"_{q_tag}_hyp{hyp_tag}_src-1_wl+1.h5")
TWOPT_FIT = Path("/Users/mcp3270/2pt_plot_meff/twopt_fit_results.h5")          # local 2pt fit results
TWOPT_TAG_BASE = "nstate3_t3-18_svd1e-12_e0prior"     # must match `tag` in your local 2pt fit
TWOPT_TAG = TWOPT_TAG_BASE + ("_fold" if FOLD_2PT else "")
OUTPUT_DIR = LOCAL / f"bare_matrix_element_{frame}_{GAMMA}_method{VACUUM_METHOD}_{DIRECTION}_2pt_{TWOPT_TAG_BASE}_perjk"
PLOT_DIR = LOCAL / f"ratio_fit_plots_perjk_{frame}_{GAMMA}_method{VACUUM_METHOD}_{DIRECTION}_{TWOPT_TAG_BASE}"

hyp_list = [8]
pf_list = [(0, 0, pz) for pz in range(0, 7)]
#pf_list = [(0,0,1)]
q_list = [(0, 0, 0)]

w_fit_list = list(range(0, 10))
tsep_fit_list = [ 6,7, 8, 9,10]
tau_skip = 3
w_plot_list = [0, 1,2,3, 4,5, 6, 8]

DE_WIDTH_FACTOR = 1.0
A_PRIOR_WIDTH = 1e4
M00_PRIOR_WIDTH = 1e4
CROSS_W_FACTOR = 1.0   # 1.0 = full joint fit, 0.0 = block diagonal in w
                       # (exactly equivalent to fitting each w separately)
SVDCUT = 1e-07
MAXIT = 20000
N_WORKERS = 7

tag = (f"tsep{tsep_fit_list[0]}-{tsep_fit_list[-1]}_svd{SVDCUT:.0e}"
       f"_dEf{DE_WIDTH_FACTOR:g}_xw{CROSS_W_FACTOR:g}_tauskip{tau_skip}"
       + ("_fold2" if FOLD_2PT else "")).replace(".", "p")

M00_START = {
    0: [-0.121, -0.116, -0.105, -0.090, -0.074, -0.059, -0.047, -0.037, -0.028, -0.021],
    1: [-0.211, -0.203, -0.182, -0.153, -0.123, -0.095, -0.070, -0.050, -0.034, -0.021],
    2: [-0.320, -0.308, -0.274, -0.228, -0.180, -0.135, -0.096, -0.065, -0.040, -0.022],
    3: [-0.374, -0.359, -0.318, -0.263, -0.205, -0.152, -0.108, -0.074, -0.049, -0.031],
    4: [-0.417, -0.400, -0.353, -0.289, -0.222, -0.161, -0.111, -0.073, -0.045, -0.027],
    5: [-0.443, -0.423, -0.374, -0.310, -0.244, -0.187, -0.142, -0.108, -0.082, -0.062],
    6: [-0.228, -0.217, -0.189, -0.153, -0.117, -0.084, -0.059, -0.042, -0.032, -0.023],
}


def make_initial(pz, prior):
    """starting point for lsqfit: prior means everywhere, M00 from the table"""
    p0 = {k: gv.mean(prior[k]) for k in prior}      # keys as stored, i.e. 'log(dEi)' not 'dEi'
    p0["M00"] = np.array(M00_START[pz])
    return p0


def jk(values, jack_axis=0):
    """Jackknife mean and error over jack_axis."""
    v = np.moveaxis(values, jack_axis, 0)
    n = v.shape[0]
    mean = v.mean(axis=0)
    return mean, np.sqrt((n - 1.0) / n * np.sum((v - mean) ** 2, axis=0))

def ratio_model(x, p):
    """input data-poins x = (tsep,tau,w) and prior; output value of that data-point"""
    w, tau, tsep_tau = x["w_index"], x["tau"], x["tsep"] - x["tau"]
    dEi = p["dEi"][w]
    dEf = p["dEf"][w] if "log(dEf)" in p else dEi
    Ai = p["Ai"][w]
    Af = p["Af"][w] if "Af" in p else Ai
    return (p["M00"][w] + Ai * gv.exp(-dEi * tau) + Af * gv.exp(-dEf * tsep_tau)
            + p["Afi"][w] * gv.exp(-dEi * tau - dEf * tsep_tau))


def make_prior(n_w, forward, gap_i=0.6,width_i=1,gap_f=0.6, width_f=1):

    """n_w : the number of z's
       forward : controls if we have Af
       gap : central value of dE from two-point fit
       width :  width of dE from two-point fit
    """


    prior = gv.BufferDict()
    prior["M00"] = gv.gvar([0.0] * n_w, [M00_PRIOR_WIDTH] * n_w)
    prior["Ai"]  = gv.gvar([0.0] * n_w, [A_PRIOR_WIDTH] * n_w)
    prior["Afi"] = gv.gvar([0.0] * n_w, [A_PRIOR_WIDTH] * n_w)
    prior["log(dEi)"] = gv.log(gv.gvar([gap_i] * n_w, [width_i*DE_WIDTH_FACTOR] * n_w))
    if not forward:
        prior["Af"] = gv.gvar([0.0] * n_w, [A_PRIOR_WIDTH] * n_w)
        prior["log(dEf)"] = gv.log(gv.gvar([gap_f] * n_w, [width_f*DE_WIDTH_FACTOR] * n_w))
    return prior


def fit_one_point(hyp, pf, q):
    label = f"hyp{hyp} pf{pf} q{q}"
    # ---- the ratio of this (hyp, pf, q) from the consolidated ratio file ----
    with h5py.File(RATIO_FILE, "r") as f:
        hyp_all = [int(v) for v in f["hyp_list"][:]]
        w_all = [int(v) for v in f["w_list"][:]]
        pf_all = [tuple(int(v) for v in p) for p in f["pf_list"][:]]
        q_all = [tuple(int(v) for v in qq) for qq in f["q_list"][:]]
        ihyp, ipf, iq = hyp_all.index(hyp), pf_all.index(tuple(pf)), q_all.index(tuple(q))
        ratio_jk = f["ratio_jk"][:, ihyp, :, iq, :, :, ipf]        # [jk, w, tsep, tau]   tsep, tau = time index
    tsep_all = list(range(ratio_jk.shape[2]))

    w_index, tau_all, T_all, vector_data = [], [], [], []

    #The following loops build the data-points to be used
    for iw_fit, w in enumerate(w_fit_list):
        for tsep in tsep_fit_list:
            for tau in range(tau_skip, tsep - tau_skip + 1):
                w_index.append(iw_fit)
                tau_all.append(float(tau))
                T_all.append(float(tsep))
                #the ratio file already holds the real part of one operator
                vector_data.append(ratio_jk[:, w_all.index(w), tsep_all.index(tsep), tau])

    #x is the dictionary of coordinates of used data-points
    x = {"w_index": np.array(w_index), "tau": np.array(tau_all),"tsep": np.array(T_all)}

    samples = np.stack(vector_data, axis=1) #(jackknife_index,data_point_index)
    n_jk, n_pt = samples.shape
    n_w = len(w_fit_list)

    # gap priors from the two-point fit at pf and at pi = pf + q, per jackknife sample
    forward = all(c == 0 for c in q)
    pi = tuple(int(pf[k] + q[k]) for k in range(3))
    with h5py.File(TWOPT_FIT, "r") as f:
        g = f[TWOPT_TAG]
        gf = g[f"p{pf[0]}_{pf[1]}_{pf[2]}"]
        gi = gf if forward else g[f"p{pi[0]}_{pi[1]}_{pi[2]}"]
        gap_f_jk = gf["dE_jk"][:, 1]
        gap_i_jk = gi["dE_jk"][:, 1]
        gap_f_err_jk = gf["dE_sdev_jk"][:, 1]  #width is per-jackknif-samplee
        gap_i_err_jk = gi["dE_sdev_jk"][:, 1]
    gap_f_mean, gap_f_err = jk(gap_f_jk)
    gap_i_mean, gap_i_err = jk(gap_i_jk)
    if len(gap_f_jk) != n_jk:
        print(f"  WARNING {label}: 2pt has {len(gap_f_jk)} samples, "
              f"ratio has {n_jk}", flush=True)
    print(f"{label}: {n_pt} points, {n_jk} samples, 2pt gap "
          f"pi{pi} {gap_i_mean:.3f} +- {gap_i_err:.3f}, "
          f"pf{tuple(pf)} {gap_f_mean:.3f} +- {gap_f_err:.3f}, "
          f"prior widths {DE_WIDTH_FACTOR * gap_i_err:.3f} / "
          f"{DE_WIDTH_FACTOR * gap_f_err:.3f}", flush=True)

    # prepare the central value of data points and covariance matrix
    mean = samples.mean(axis=0)
    cov = np.cov(samples, rowvar=False, ddof=1) * (n_jk - 1.0) ** 2 / n_jk #with ddof=1, already a n_jk-1 factor is in the denominator
    same_w = x["w_index"][:, None] == x["w_index"][None, :]

    #np.where(condition,a,b): At slots where condition is true(1), take value from a, if faluse(0). take value from b)
    cov = np.where(same_w, cov, CROSS_W_FACTOR * cov)

    central_fit = lsqfit.nonlinear_fit(
        data=(x, gv.gvar(mean, cov)), fcn=ratio_model, svdcut=SVDCUT,
        maxit=MAXIT, prior=make_prior(n_w,  forward, gap_i_mean, gap_i_err,gap_f_mean,gap_f_err))
    print(f"For p={pf}, dE from ratio central fit {np.exp(central_fit.pmean['log(dEi)'])}")

    if getattr(central_fit, "svdn", 0):
        print(f"  WARNING {label}: svdcut modified {central_fit.svdn} of {n_pt} "
                f"modes", flush=True)
    p0 = {k: central_fit.pmean[k] for k in central_fit.prior} # build a dictionary of central values of fit parameters

    names = ["M00", "Ai", "Afi", "dEi"] + ([] if forward else ["Af", "dEf"])
    params = {n: np.zeros((n_jk, n_w)) for n in names}
    chi2dof, Q = np.zeros(n_jk), np.zeros(n_jk)
    for i in range(n_jk):
        prior_jk=make_prior(n_w, forward, gap_i_jk[i], gap_i_err_jk[i],gap_f_jk[i], gap_f_err_jk[i])
        jackknife_fit = lsqfit.nonlinear_fit(
            data=(x, gv.gvar(samples[i], cov)), fcn=ratio_model,
            svdcut=SVDCUT, maxit=MAXIT,
            prior=prior_jk, p0=p0)
        for n in names:
            params[n][i] = gv.mean(jackknife_fit.p[n]) #gv.mean(array of gvar objects) = array of central values
        chi2dof[i], Q[i] = jackknife_fit.chi2 / jackknife_fit.dof, jackknife_fit.Q

    M00_mean, M00_err = jk(params["M00"])
    out = OUTPUT_DIR / (f"bareM_{operator}_{frame}_hyp{hyp}"
                        f"_pf{pf[0]}_{pf[1]}_{pf[2]}"
                        f"_q{q[0]}_{q[1]}_{q[2]}_{tag}.h5")
    with h5py.File(out, "w") as f:
        f.create_dataset("bare_matrix_element_jk", data=params["M00"])
        for n in names:
            f.create_dataset(f"{n}_jk", data=params[n])
        f.create_dataset("M00_jk_mean", data=M00_mean)
        f.create_dataset("M00_jk_err", data=M00_err)
        f.create_dataset("w_list", data=np.array(w_fit_list, dtype=np.int64))
        f.create_dataset("gap_i_prior_center_jk", data=gap_i_jk)
        f.create_dataset("gap_f_prior_center_jk", data=gap_f_jk)
        f.attrs["dim_bare_matrix_element_jk"] = "jk,w_index"
        f.attrs["operator"] = operator
        f.attrs["twopt_tag"] = TWOPT_TAG
        f.attrs["frame"] = frame
        f.attrs["hyp"] = hyp
        f.attrs["pf"] = np.array(pf, dtype=np.int64)
        f.attrs["q"] = np.array(q, dtype=np.int64)
        f.attrs["gap_prior"] = (f"2pt E1-E0 per sample, width "
                                f"{DE_WIDTH_FACTOR} x its jackknife error")
        f.attrs["gap_i_prior_width"] = DE_WIDTH_FACTOR * gap_i_err
        f.attrs["gap_f_prior_width"] = DE_WIDTH_FACTOR * gap_f_err
        f.attrs["pi"] = np.array(pi, dtype=np.int64)
        f.attrs["tsep_fit_list"] = np.array(tsep_fit_list, dtype=np.int64)
        f.attrs["tau_skip"] = tau_skip
        f.attrs["svdcut"] = SVDCUT
        f.attrs["cross_w_factor"] = CROSS_W_FACTOR
        f.attrs["fold_2pt"] = FOLD_2PT
        f.attrs["gamma"] = GAMMA
        f.attrs["vacuum_method"] = VACUUM_METHOD
        f.attrs["direction"] = DIRECTION
        f.attrs["ratio_source"] = str(RATIO_FILE)
        f.attrs["svdn"] = int(getattr(central_fit, "svdn", 0))
        f.attrs["chi2dof"] = float(chi2dof.mean())
        f.attrs["Q"] = float(Q.mean())

    plot_fit(x, samples, params, chi2dof.mean(), Q.mean(), hyp, pf, q)
    print(f"  saved {out.name}  <chi2/dof> {chi2dof.mean():.2f}", flush=True)


def plot_fit(x, samples, params, chi2dof, q_value, hyp, pf, q):
    data_mean, data_err = jk(samples)
    m00_mean, m00_err = jk(params["M00"])
    ws = [w for w in w_plot_list if w in w_fit_list]
    fig, axes = plt.subplots(1, len(ws), figsize=(4.2 * len(ws), 4.6),
                             squeeze=False)
    for icol, w in enumerate(ws):
        ax, iw = axes[0][icol], w_fit_list.index(w)
        for itsep, tsep in enumerate(tsep_fit_list):
            sel = (x["w_index"] == iw) & (x["tsep"] == tsep)
            ax.errorbar(x["tau"][sel] - 0.5 * tsep, data_mean[sel],
                        yerr=data_err[sel], marker="o", ls="none", ms=4,
                        capsize=3, color=f"C{itsep}", label=f"tsep={tsep}")
            # the band: the fitted curve on every jackknife sample
            tau = np.linspace(tau_skip, tsep - tau_skip, 60)[None, :]
            ai = params["Ai"][:, iw][:, None]
            de = params["dEi"][:, iw][:, None]
            af = params.get("Af", params["Ai"])[:, iw][:, None]
            df = params.get("dEf", params["dEi"])[:, iw][:, None]
            curve = (params["M00"][:, iw][:, None] + ai * np.exp(-de * tau)
                     + af * np.exp(-df * (tsep - tau))
                     + params["Afi"][:, iw][:, None]
                     * np.exp(-de * tau - df * (tsep - tau)))
            b_mean, b_err = jk(curve)
            ax.fill_between(tau[0] - 0.5 * tsep, b_mean - b_err,
                            b_mean + b_err, color=f"C{itsep}", alpha=0.25,
                            lw=0)
        ax.axhspan(m00_mean[iw] - m00_err[iw], m00_mean[iw] + m00_err[iw],
                   color="k", alpha=0.15)
        ax.set_title(f"w = {w}   M00 = {gv.gvar(m00_mean[iw], m00_err[iw])}",
                     fontsize=10)
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$")
        ax.set_ylim(-0.6,0.1)
    axes[0][0].set_ylabel("R")
    axes[0][-1].legend(fontsize=8)
    fig.suptitle(f"{operator}  {GAMMA}  method{VACUUM_METHOD}  {DIRECTION}  pf={pf}  q={q}  hyp={hyp}  "
                 f"<chi2/dof>={chi2dof:.2f}  <Q>={q_value:.2f}", fontsize=12)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / (f"fit_{operator}_{frame}_hyp{hyp}"
                            f"_pf{pf[0]}_{pf[1]}_{pf[2]}"
                            f"_q{q[0]}_{q[1]}_{q[2]}_{tag}_{DIRECTION}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    tasks = [(hyp, pf, q) for hyp in hyp_list for pf in pf_list
             for q in q_list]
    print(f"{len(tasks)} fits from {RATIO_FILE.name}, N_WORKERS = {N_WORKERS}", flush=True)
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(fit_one_point, *t) for t in tasks]
        for future in as_completed(futures):
            future.result()
    print("finished", flush=True)

Overwriting fit_ratio_hyp.py


In [61]:
!python3 fit_ratio_hyp.py


7 fits from ratio_G45_method1_fb_TXTXpTYTYm2XYXY_q0_0_0_hyp0-1-2-3-4-5-6-7-8_src-1_wl+1.h5, N_WORKERS = 7
hyp8 pf(0, 0, 1) q(0, 0, 0): 150 points, 799 samples, 2pt gap pi(0, 0, 1) 0.393 +- 0.021, pf(0, 0, 1) 0.393 +- 0.021, prior widths 0.021 / 0.021
hyp8 pf(0, 0, 2) q(0, 0, 0): 150 points, 799 samples, 2pt gap pi(0, 0, 2) 0.293 +- 0.022, pf(0, 0, 2) 0.293 +- 0.022, prior widths 0.022 / 0.022
hyp8 pf(0, 0, 0) q(0, 0, 0): 150 points, 799 samples, 2pt gap pi(0, 0, 0) 0.639 +- 0.111, pf(0, 0, 0) 0.639 +- 0.111, prior widths 0.111 / 0.111
hyp8 pf(0, 0, 5) q(0, 0, 0): 150 points, 799 samples, 2pt gap pi(0, 0, 5) 0.245 +- 0.044, pf(0, 0, 5) 0.245 +- 0.044, prior widths 0.044 / 0.044
hyp8 pf(0, 0, 4) q(0, 0, 0): 150 points, 799 samples, 2pt gap pi(0, 0, 4) 0.261 +- 0.030, pf(0, 0, 4) 0.261 +- 0.030, prior widths 0.030 / 0.030
hyp8 pf(0, 0, 3) q(0, 0, 0): 150 points, 799 samples, 2pt gap pi(0, 0, 3) 0.265 +- 0.027, pf(0, 0, 3) 0.265 +- 0.027, prior widths 0.027 / 0.027
hyp8 pf(0, 0, 6) q(0, 0,

In [62]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

LOCAL = Path("/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis")
TWOPT_FIT = Path("/Users/mcp3270/2pt_plot_meff/twopt_fit_results.h5")
TWOPT_TAG = "nstate3_t3-18_svd1e-12_e0prior"
GAMMA = "G45"
method = 1
operator = "TXTXpTYTYm2XYXY"
frame = "symmetric"
hyp = 8
bareM_tag = "tsep6-10_svd1e-07_dEf1_xw1_tauskip3"      # must equal the fitter's `tag`
pf_list = [(0, 0, pz) for pz in range(0, 7)]
q = (0, 0, 0)
q_tag = "q" + "_".join(str(v) for v in q)
Ls = 32
PF_REF, W_REF = (0, 0, 0), 0
directions = ["fb"]                                    # "forward", "fb", or both
dir_tag = "_".join(directions)
PLOT_DIR = LOCAL / "itd_plots"
PLOT_DIR.mkdir(exist_ok=True)


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


# ground-state energy of every pf from the 2pt fit, [jk, pf]
with h5py.File(TWOPT_FIT, "r") as f:
    g = f[TWOPT_TAG]
    E = np.stack([g[f"p{p[0]}_{p[1]}_{p[2]}"]["E_jk"][:, 0] for p in pf_list], axis=1)

# bare matrix elements per direction, [jk, pf, w], and the two ratios formed sample by sample
M, single, double = {}, {}, {}
for d in directions:
    blocks = []
    for pf in pf_list:
        name = (f"bareM_{operator}_{frame}_hyp{hyp}_pf{pf[0]}_{pf[1]}_{pf[2]}"
                f"_q{q[0]}_{q[1]}_{q[2]}_{bareM_tag}.h5")
        folder = LOCAL / f"bare_matrix_element_{frame}_{GAMMA}_method{method}_{d}_2pt_{TWOPT_TAG}_perjk"
        with h5py.File(folder / name, "r") as f:
            blocks.append(f["bare_matrix_element_jk"][:].real)
            w_list = f["w_list"][:]
    M[d] = np.stack(blocks, axis=1)                                                     # [jk, pf, w]
    ipf, iw = pf_list.index(PF_REF), list(w_list).index(W_REF)
    kinematic = E[:, ipf][:, None] / E                                                  # E(pf_ref) / E(pf), [jk, pf]
    single[d] = M[d] / M[d][:, ipf:ipf + 1, :] * kinematic[:, :, None]                 # [jk, pf, w]
    double[d] = (M[d] / M[d][:, :, iw:iw + 1]) / (M[d][:, ipf:ipf + 1, :] / M[d][:, ipf:ipf + 1, iw:iw + 1])

pz = np.array([p[2] for p in pf_list], dtype=float)
nu = (2 * np.pi / Ls) * pz[:, None] * w_list[None, :]                                  # [pf, w]

# ---------- table: values, and the comparison only if both directions were read ----------
print(f"{GAMMA}  method {method}  hyp{hyp}  {bareM_tag}   directions: {', '.join(directions)}")
if len(directions) == 2:
    print(f"{'pz':>3} {'w':>2} {'nu':>6} | {'double fwd':>16} {'double fb':>16} {'diff/err':>8} {'err fb/fwd':>10}"
          f" | {'single fwd':>16} {'single fb':>16} {'diff/err':>8} {'err fb/fwd':>10}")
    for i, pf in enumerate(pf_list[1:], start=1):
        for w in [1, 2, 3, 4, 6, 8]:
            j = list(w_list).index(w)
            df, edf = jk(double["forward"][:, i, j]); db, edb = jk(double["fb"][:, i, j])
            sf, esf = jk(single["forward"][:, i, j]); sb, esb = jk(single["fb"][:, i, j])
            dd, _ = jk(double["fb"][:, i, j] - double["forward"][:, i, j])
            ds, _ = jk(single["fb"][:, i, j] - single["forward"][:, i, j])
            print(f"{int(pz[i]):>3} {w:>2} {nu[i, j]:6.2f} | {df:7.3f} +- {edf:5.3f} {db:7.3f} +- {edb:5.3f} {dd/edf:+8.2f} {edb/edf:10.2f}"
                  f" | {sf:7.3f} +- {esf:5.3f} {sb:7.3f} +- {esb:5.3f} {ds/esf:+8.2f} {esb/esf:10.2f}")
else:
    d = directions[0]
    print(f"{'pz':>3} {'w':>2} {'nu':>6} | {'double':>16} | {'single':>16}")
    for i, pf in enumerate(pf_list[1:], start=1):
        for w in [1, 2, 3, 4]:
            j = list(w_list).index(w)
            dm, de = jk(double[d][:, i, j]); sm, se = jk(single[d][:, i, j])
            print(f"{int(pz[i]):>3} {w:>2} {nu[i, j]:6.2f} | {dm:7.3f} +- {de:5.3f} | {sm:7.3f} +- {se:5.3f}")

# ---------- figure: bare M00, single ratio, double ratio ----------
style = {"forward": dict(marker="o", ls="-", lw=0.8, ms=4),
         "fb":      dict(marker="o", mfc="none", ls="--", lw=0.8, ms=6, mew=1.2)}
legend_text = ",  ".join("filled forward" if d == "forward" else "hollow forward+backward" for d in directions)
plt.figure(figsize=(18, 5.2))
plt.subplot(1, 3, 1)
for i, pf in enumerate(pf_list):
    for d in directions:
        mean, err = jk(M[d][:, i, :])
        plt.errorbar(w_list, mean, yerr=err, capsize=2, color=f"C{i}",
                     label=f"pz={int(pz[i])}" if d == directions[0] else None, **style[d])
plt.xlabel("w")
plt.ylabel("bare M00")
plt.title("bare matrix element:  " + legend_text, fontsize=9)
plt.legend(fontsize=8, ncol=2)
for k, (data, title, ylim) in enumerate([(single, r"single ratio $\times E_{\rm ref}/E$", (-0.2, 1.6)),
                                         (double, "double ratio", (-0.2, 1.2))], start=2):
    plt.subplot(1, 3, k)
    for i, pf in enumerate(pf_list[1:], start=1):
        for d in directions:
            mean, err = jk(data[d][:, i, :])
            plt.errorbar(nu[i], mean, yerr=err, capsize=2, color=f"C{i}",
                         label=f"pz={int(pz[i])}" if d == directions[0] else None, **style[d])
    plt.axhline(1.0, color="k", ls=":", lw=1, alpha=0.5)
    plt.xlabel(r"$\nu = 2\pi p_z w / L_s$")
    plt.title(title + ":  " + legend_text, fontsize=9)
    plt.ylim(*ylim)
    plt.xlim(-0.1, 11)
    if k == 2:
        plt.legend(fontsize=8, ncol=2)
plt.suptitle(f"{operator}   {GAMMA}   method {method}   hyp{hyp}   {q_tag}   {bareM_tag}", fontsize=11)
plt.tight_layout()
out = PLOT_DIR / f"itd_{dir_tag}_{GAMMA}_method{method}_{operator}_{q_tag}_hyp{hyp}_{bareM_tag}.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print("saved", out)

G45  method 1  hyp8  tsep6-10_svd1e-07_dEf1_xw1_tauskip3   directions: fb
 pz  w     nu |           double |           single
  1  1   0.20 |   1.007 +- 0.006 |   1.303 +- 0.113
  1  2   0.39 |   1.018 +- 0.022 |   1.316 +- 0.123
  1  3   0.59 |   1.018 +- 0.048 |   1.316 +- 0.142
  1  4   0.79 |   0.981 +- 0.080 |   1.269 +- 0.165
  2  1   0.39 |   1.017 +- 0.008 |   1.475 +- 0.173
  2  2   0.79 |   1.051 +- 0.031 |   1.524 +- 0.194
  2  3   1.18 |   1.077 +- 0.070 |   1.562 +- 0.229
  2  4   1.57 |   1.051 +- 0.117 |   1.524 +- 0.270
  3  1   0.59 |   1.026 +- 0.011 |   1.273 +- 0.174
  3  2   1.18 |   1.084 +- 0.043 |   1.345 +- 0.200
  3  3   1.77 |   1.140 +- 0.098 |   1.415 +- 0.247
  3  4   2.36 |   1.125 +- 0.160 |   1.397 +- 0.299
  4  1   0.79 |   1.012 +- 0.013 |   1.104 +- 0.198
  4  2   1.57 |   1.023 +- 0.045 |   1.116 +- 0.211
  4  3   2.36 |   0.996 +- 0.092 |   1.087 +- 0.229
  4  4   3.14 |   0.886 +- 0.136 |   0.966 +- 0.239
  5  1   0.98 |   0.996 +- 0.018 |   0.947

In [1]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

LOCAL = Path("/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis")
TWOPT_FIT = Path("/Users/mcp3270/2pt_plot_meff/twopt_fit_results.h5")
TWOPT_TAG = "nstate3_t3-18_svd1e-12_e0prior"
GAMMA = "G45"
method = 1
operator = "TXTXpTYTYm2XYXY"
frame = "symmetric"
hyp = 8                                                # the level shown as filled points
hyp_hollow = 8                                         # the level shown as hollow points
hyp_compare_list = [0, 1, 2, 3, 4, 5, 6, 7, 8]         # levels listed in the table; all need fitter output
bareM_tag = "tsep5-10_svd1e-07_dEf1_xw1_tauskip2"      # must equal the fitter's `tag`
pf_list = [(0, 0, pz) for pz in range(0, 7)]
q = (0, 0, 0)
q_tag = "q" + "_".join(str(v) for v in q)
Ls = 32
PF_REF, W_REF = (0, 0, 0), 0
direction = "fb"                                       # "forward", "backward" or "fb"
PLOT_DIR = LOCAL / "itd_plots"
PLOT_DIR.mkdir(exist_ok=True)
hyp_used = sorted(set(hyp_compare_list) | {hyp, hyp_hollow})


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def load_bareM(h):
    """bare matrix elements at one HYP level, [jk, pf, w]"""
    blocks = []
    for pf in pf_list:
        name = (f"bareM_{operator}_{frame}_hyp{h}_pf{pf[0]}_{pf[1]}_{pf[2]}"
                f"_q{q[0]}_{q[1]}_{q[2]}_{bareM_tag}.h5")
        folder = LOCAL / f"bare_matrix_element_{frame}_{GAMMA}_method{method}_{direction}_2pt_{TWOPT_TAG}_perjk"
        with h5py.File(folder / name, "r") as f:
            blocks.append(f["bare_matrix_element_jk"][:].real)
            w = f["w_list"][:]
    return np.stack(blocks, axis=1), w


# ground-state energy of every pf from the 2pt fit, [jk, pf]
with h5py.File(TWOPT_FIT, "r") as f:
    g = f[TWOPT_TAG]
    E = np.stack([g[f"p{p[0]}_{p[1]}_{p[2]}"]["E_jk"][:, 0] for p in pf_list], axis=1)

# bare matrix element and the two ratios at every level, [hyp] -> [jk, pf, w]
M, single, double = {}, {}, {}
for h in hyp_used:
    Mh, w_list = load_bareM(h)
    ipf, iw = pf_list.index(PF_REF), list(w_list).index(W_REF)
    kinematic = E[:, ipf][:, None] / E                                                # E(pf_ref) / E(pf), [jk, pf]
    M[h] = Mh                                                                         # [jk, pf, w]
    single[h] = Mh / Mh[:, ipf:ipf + 1, :] * kinematic[:, :, None]                    # [jk, pf, w]
    double[h] = (Mh / Mh[:, :, iw:iw + 1]) / (Mh[:, ipf:ipf + 1, :] / Mh[:, ipf:ipf + 1, iw:iw + 1])

pz = np.array([p[2] for p in pf_list], dtype=float)
nu = (2 * np.pi / Ls) * pz[:, None] * w_list[None, :]                                 # [pf, w]

# ---------- table: double and single ratio at every level in hyp_compare_list ----------
print(f"{GAMMA}  method {method}  direction {direction}  {q_tag}  {bareM_tag}")
print(f"{'pz':>3} {'w':>2} {'nu':>6} | " + " ".join(f"{'double hyp'+str(h):>16}" for h in hyp_compare_list)
      + " | " + " ".join(f"{'single hyp'+str(h):>16}" for h in hyp_compare_list))
for i, pf in enumerate(pf_list[1:], start=1):
    for w in [1, 2, 3, 4, 6, 8]:
        j = list(w_list).index(w)
        line = f"{int(pz[i]):>3} {w:>2} {nu[i, j]:6.2f} | "
        for h in hyp_compare_list:
            dm, de = jk(double[h][:, i, j])
            line += f"{dm:8.3f} +-{de:6.3f} "
        line += "| "
        for h in hyp_compare_list:
            sm, se = jk(single[h][:, i, j])
            line += f"{sm:8.3f} +-{se:6.3f} "
        print(line)

# ---------- figure: bare M00, single ratio, double ratio; filled = hyp, hollow = hyp_hollow ----------
plt.figure(figsize=(18, 5.2))
plt.subplot(1, 3, 1)
for i, pf in enumerate(pf_list):
    mean, err = jk(M[hyp][:, i, :])
    plt.errorbar(w_list, mean, yerr=err, marker="o", ls="-", lw=0.8, ms=4, capsize=2,
                 color=f"C{i}", label=f"pz={int(pz[i])}")
    mean, err = jk(M[hyp_hollow][:, i, :])
    plt.errorbar(w_list + 0.1, mean, yerr=err, marker="o", mfc="none", ls="--", lw=0.8, ms=6,
                 mew=1.2, capsize=2, color=f"C{i}")
plt.xlabel("w")
plt.ylabel("bare M00")
plt.title(f"bare matrix element:  filled hyp{hyp},  hollow hyp{hyp_hollow}", fontsize=9)
plt.legend(fontsize=8, ncol=2)
for k, (data, title, ylim) in enumerate([(single, r"single ratio $\times E_{\rm ref}/E$", (-0.2, 1.6)),
                                         (double, "double ratio", (-0.2, 1.2))], start=2):
    plt.subplot(1, 3, k)
    for i, pf in enumerate(pf_list[1:], start=1):
        mean, err = jk(data[hyp][:, i, :])
        plt.errorbar(nu[i], mean, yerr=err, marker="o", ls="-", lw=0.8, ms=4, capsize=2,
                     color=f"C{i}", label=f"pz={int(pz[i])}")
        mean, err = jk(data[hyp_hollow][:, i, :])
        plt.errorbar(nu[i] + 0.06, mean, yerr=err, marker="o", mfc="none", ls="--", lw=0.8, ms=6,
                     mew=1.2, capsize=2, color=f"C{i}")
    plt.axhline(1.0, color="k", ls=":", lw=1, alpha=0.5)
    plt.xlabel(r"$\nu = 2\pi p_z w / L_s$")
    plt.title(title + f":  filled hyp{hyp},  hollow hyp{hyp_hollow}", fontsize=9)
    plt.ylim(*ylim)
    plt.xlim(-0.1, 11)
    if k == 2:
        plt.legend(fontsize=8, ncol=2)
plt.suptitle(f"{operator}   {GAMMA}   method {method}   {direction}   {q_tag}   {bareM_tag}   hyp{hyp} vs hyp{hyp_hollow}", fontsize=11)
plt.tight_layout()
out = PLOT_DIR / f"itd_hypcompare_{direction}_{GAMMA}_method{method}_{operator}_{q_tag}_hyp{hyp}_vs_hyp{hyp_hollow}_{bareM_tag}.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print("saved", out)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/bare_matrix_element_symmetric_G45_method1_fb_2pt_nstate3_t3-18_svd1e-12_e0prior_perjk/bareM_TXTXpTYTYm2XYXY_symmetric_hyp0_pf0_0_0_q0_0_0_tsep5-10_svd1e-07_dEf1_xw1_tauskip2.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)